# Experiment 8: Abstractive Text Summarization using mT5 and mBART

**Platform:** Google Colab  
**Dataset:** XL-Sum (Cross-Lingual Summarization)  
**Language selected:** Hindi  
**Models:**
1. mT5 — Multilingual Text-to-Text Transfer Transformer (`google/mt5-small`)
2. mBART-50 — Multilingual Bidirectional and Auto-Regressive Transformer (`facebook/mbart-large-50`)

**Evaluation metrics:** ROUGE-1, ROUGE-2, ROUGE-L and multilingual BERTScore (mBERT-based).

This notebook fine-tunes both models on the same Hindi XL-Sum training subset and evaluates them on the same held-out test subset. The XL-Sum data is loaded from its published Parquet files, which is compatible with current Hugging Face Datasets releases.

**Colab recommendation:** Select `Runtime → Change runtime type → T4 GPU`. Fine-tuning mBART-50 is resource-intensive. The default subsets are deliberately small for a practical demonstration; increase them if you have sufficient GPU memory and runtime. Run cells in order.

## 1. Install required libraries

We use Hugging Face Datasets and Transformers for the dataset and sequence-to-sequence models, and Evaluate for ROUGE and BERTScore.

In [ ]:
!pip -q install -U transformers datasets evaluate accelerate sentencepiece rouge_score bert-score

In [ ]:
import os, random, numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
)
import evaluate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU - use Colab GPU runtime")

## 2. Load the Hindi XL-Sum dataset

XL-Sum provides news articles (`text`) and human-written summaries (`summary`). We use the Hindi configuration as the selected language for both models.

In [ ]:
# Load the Hindi XL-Sum Parquet files directly.
# This avoids legacy dataset-script loading, which newer `datasets` versions no longer support.
from datasets import load_dataset

BASE = "https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/main/hindi/"
data_files = {
    "train": [
        BASE + "xlsum-train-00000-of-00002.parquet",
        BASE + "xlsum-train-00001-of-00002.parquet"
    ],
    "validation": BASE + "xlsum-validation.parquet",
    "test": BASE + "xlsum-test.parquet"
}

raw_dataset = load_dataset("parquet", data_files=data_files)

print(raw_dataset)
print("Available splits:", list(raw_dataset.keys()))
print("Dataset columns:", raw_dataset["train"].column_names)
print("\\nExample article:")
print(raw_dataset["train"][0]["text"][:1000])
print("\\nReference summary:")
print(raw_dataset["train"][0]["summary"])

## 3. Prepare train, validation and test subsets

The official Hindi XL-Sum Parquet files provide separate train, validation and test splits. To keep the Colab experiment manageable, the notebook selects fixed-size subsets from these splits. Both models use exactly the same examples. Set a limit to `None` to use the complete corresponding split, subject to available compute.

In [ ]:
TRAIN_LIMIT = 500   # set None for full train split
VALID_LIMIT = 100   # set None for full validation split
TEST_LIMIT  = 100   # set None for full test split

def take_subset(split, limit, seed=SEED):
    if limit is None or limit >= len(split):
        return split
    return split.shuffle(seed=seed).select(range(limit))

train_data = take_subset(raw_dataset["train"], TRAIN_LIMIT)
valid_data = take_subset(raw_dataset["validation"], VALID_LIMIT)
test_data  = take_subset(raw_dataset["test"], TEST_LIMIT)

print("Train:", len(train_data))
print("Validation:", len(valid_data))
print("Test:", len(test_data))

## 4. Common preprocessing and evaluation setup

The input is the Hindi news article and the target is its reference summary. We truncate long articles and summaries to fixed token limits. ROUGE measures n-gram and sequence overlap. BERTScore uses contextual multilingual BERT representations to compare generated and reference summaries semantically.

In [ ]:
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128

rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def compute_summary_metrics(predictions, references):
    # ROUGE
    rouge = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=False
    )

    # Multilingual BERTScore for Hindi
    bert = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        lang="hi",
        model_type="bert-base-multilingual-cased",
        batch_size=8,
        verbose=False
    )

    return {
        "ROUGE-1": float(rouge["rouge1"]),
        "ROUGE-2": float(rouge["rouge2"]),
        "ROUGE-L": float(rouge["rougeL"]),
        "mBERT Precision": float(np.mean(bert["precision"])),
        "mBERT Recall": float(np.mean(bert["recall"])),
        "mBERT F1": float(np.mean(bert["f1"]))
    }

## 5. Shared fine-tuning and generation function

The function below prepares tokenized data, fine-tunes a sequence-to-sequence model, generates summaries for the test articles, and calculates all required metrics. It is reused for mT5 and mBART to keep the experimental procedure consistent.

In [ ]:
def run_summarization_experiment(
    model_name,
    run_name,
    tokenizer_kwargs=None,
    model_kwargs=None,
    forced_bos_token_id=None
):
    tokenizer_kwargs = tokenizer_kwargs or {}
    model_kwargs = model_kwargs or {}

    print(f"\\nLoading {run_name}: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, **tokenizer_kwargs)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)

    def preprocess(batch):
        inputs = tokenizer(
            batch["text"],
            max_length=MAX_SOURCE_LENGTH,
            truncation=True
        )
        targets = tokenizer(
            text_target=batch["summary"],
            max_length=MAX_TARGET_LENGTH,
            truncation=True
        )
        inputs["labels"] = targets["input_ids"]
        return inputs

    tokenized_train = train_data.map(
        preprocess, batched=True, remove_columns=train_data.column_names
    )
    tokenized_valid = valid_data.map(
        preprocess, batched=True, remove_columns=valid_data.column_names
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer, model=model, pad_to_multiple_of=8
    )

    output_dir = f"./results_{run_name}"
    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=5e-5,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        weight_decay=0.01,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        fp16=torch.cuda.is_available(),
        logging_steps=25,
        report_to="none",
        seed=SEED,
        gradient_checkpointing=True,
        dataloader_num_workers=0
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_valid,
        processing_class=tokenizer,
        data_collator=data_collator
    )

    trainer.train()

    # Generate summaries on the same held-out test set
    model.eval()
    device = next(model.parameters()).device
    generated_summaries = []

    for start in range(0, len(test_data), 4):
        batch_articles = test_data[start:start+4]["text"]
        encoded = tokenizer(
            batch_articles,
            return_tensors="pt",
            max_length=MAX_SOURCE_LENGTH,
            truncation=True,
            padding=True
        ).to(device)

        generation_args = {
            "max_length": MAX_TARGET_LENGTH,
            "num_beams": 4,
            "early_stopping": True
        }
        if forced_bos_token_id is not None:
            generation_args["forced_bos_token_id"] = forced_bos_token_id

        with torch.no_grad():
            output_ids = model.generate(**encoded, **generation_args)

        generated_summaries.extend(
            tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        )

    reference_summaries = test_data["summary"]
    metric_results = compute_summary_metrics(
        generated_summaries, reference_summaries
    )

    print(f"\\n{run_name} evaluation:")
    for metric, value in metric_results.items():
        print(f"{metric}: {value:.4f}")

    # Show a few examples for qualitative inspection
    examples = pd.DataFrame({
        "Article": test_data["text"][:3],
        "Reference Summary": reference_summaries[:3],
        "Generated Summary": generated_summaries[:3]
    })
    display(examples)

    del trainer, model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metric_results, generated_summaries

## 6. Model 1 — mT5

mT5 is a multilingual text-to-text Transformer. It uses the same text-to-text framework for many NLP tasks. For summarization, the article is supplied as input and the summary is generated as output.

We use `google/mt5-small` for a practical Colab experiment.

In [ ]:
mt5_results, mt5_predictions = run_summarization_experiment(
    model_name="google/mt5-small",
    run_name="mt5_hindi"
)

## 7. Model 2 — mBART-50

mBART-50 is a multilingual sequence-to-sequence Transformer pretrained with a multilingual denoising objective. We use the pretrained `facebook/mbart-large-50` checkpoint and fine-tune it for Hindi summarization. The Hindi language code is `hi_IN`; we set it as the source language and force generated summaries to begin with the Hindi target-language token.

In [ ]:
MBART_MODEL = "facebook/mbart-large-50"

# mBART-50 uses language codes to control multilingual input and output.
mbart_tokenizer = AutoTokenizer.from_pretrained(
    MBART_MODEL, src_lang="hi_IN", tgt_lang="hi_IN"
)
hindi_token_id = mbart_tokenizer.lang_code_to_id["hi_IN"]

del mbart_tokenizer

mbart_results, mbart_predictions = run_summarization_experiment(
    model_name=MBART_MODEL,
    run_name="mbart_hindi",
    tokenizer_kwargs={"src_lang": "hi_IN", "tgt_lang": "hi_IN"},
    forced_bos_token_id=hindi_token_id
)

## 8. Comparative analysis of mT5 and mBART

The table compares both models using ROUGE-1, ROUGE-2, ROUGE-L and multilingual BERTScore precision, recall and F1. Higher values generally indicate better overlap or semantic similarity for the corresponding metric.

In [ ]:
comparison = pd.DataFrame([
    {"Model": "mT5", **mt5_results},
    {"Model": "mBART", **mbart_results}
]).set_index("Model")

display(comparison.round(4))

comparison[["ROUGE-1", "ROUGE-2", "ROUGE-L", "mBERT F1"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1), rot=0
)
plt.title("mT5 vs mBART — Hindi XL-Sum Summarization")
plt.ylabel("Score")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Best model by ROUGE-L:", comparison["ROUGE-L"].idxmax())
print("Best model by mBERT F1:", comparison["mBERT F1"].idxmax())

## 9. Conclusion

Use the actual scores from your Colab execution in the practical record.

1. Abstractive summarization generates a concise summary in natural language rather than merely selecting sentences from the source.
2. mT5 performs multilingual text-to-text generation and can be fine-tuned for Hindi summarization.
3. mBART uses multilingual sequence-to-sequence pretraining and language-specific tokens to control the output language.
4. ROUGE-1, ROUGE-2 and ROUGE-L evaluate lexical overlap between generated and reference summaries.
5. Multilingual BERTScore compares contextual representations and provides a semantic similarity measure for Hindi summaries.
6. Compare the models using the measured metrics and inspect generated examples. Do not assume one model will always outperform the other.

**Experimental limitation:** The default run uses small subsets and one training epoch to fit a Colab workflow. Scores from this demonstration are not directly comparable with published XL-Sum benchmark results, which may use larger training data, more epochs, and different decoding settings. `facebook/mbart-large-50` is a large checkpoint and may run close to the GPU memory limit on a T4; if an out-of-memory error occurs, reduce `MAX_SOURCE_LENGTH` (for example, to 256) and keep per-device batch size at 1.

## 10. Viva questions and answers

**1. What is text summarization?**  
The process of generating a shorter version of a document while preserving its important information.

**2. What is abstractive summarization?**  
It generates new phrases and sentences to express the main ideas, rather than only extracting original sentences.

**3. What is XL-Sum?**  
A cross-lingual summarization dataset containing news articles and summaries in multiple languages.

**4. What is mT5?**  
A multilingual text-to-text Transformer model that treats NLP tasks as text input to text output.

**5. What is mBART?**  
A multilingual sequence-to-sequence Transformer pretrained using a denoising objective.

**6. What is the difference between mT5 and mBART?**  
They use different pretraining approaches and model/tokenization conventions. Both can be fine-tuned for multilingual summarization.

**7. What is ROUGE-1?**  
Overlap of unigrams between generated and reference summaries.

**8. What is ROUGE-2?**  
Overlap of bigrams between generated and reference summaries.

**9. What is ROUGE-L?**  
A metric based on the longest common subsequence between generated and reference text.

**10. What is BERTScore?**  
A metric that compares contextual token embeddings from a pretrained language model to estimate semantic similarity.

**11. Why use mBERT for Hindi?**  
A multilingual BERT model provides contextual representations across multiple languages, including Hindi.

**12. What is teacher forcing?**  
During training, the decoder receives the correct previous target tokens as input when predicting the next token.

**13. What is beam search?**  
A decoding strategy that keeps several candidate sequences at each generation step and selects a high-scoring sequence.

**14. Why do we compare ROUGE and BERTScore?**  
ROUGE measures lexical overlap, while BERTScore measures contextual semantic similarity. Together they provide complementary evaluation.